In [ ]:
import json

def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

In [ ]:
traindata = load_data('yo_train.json')
traindata[:5]

[{'left_context': '',
  'word': 'промокнЕм',
  'right_context': 'тесто полотенцем, чтобы убрать остатки влаги перед раскаткой.',
  'target': 1},
 {'left_context': 'Уборщик быстро',
  'word': 'смЕл',
  'right_context': 'мусор в угол комнаты.',
  'target': 1},
 {'left_context': 'На репетиции ещё неуверенно, но к концерту точно',
  'word': 'запоЕм',
  'right_context': '',
  'target': 1},
 {'left_context': 'К вечеру мы наконец добрались к древнему',
  'word': 'склЕпу',
  'right_context': 'среди кипарисов.',
  'target': 0},
 {'left_context': 'Как только нас пригласят, мы',
  'word': 'опознаЕм',
  'right_context': 'погибшего в присутствии следователя.',
  'target': 0}]

In [ ]:
testdata = load_data("yo_test.json")
testdata[:10]

[{'left_context': 'Ты наконец',
  'word': 'признаЕшься',
  'right_context': 'в своей ошибке.'},
 {'left_context': 'Господа! Давайте же',
  'word': 'запахнЕм',
  'right_context': 'плащи и выйдем на прогулку.'},
 {'left_context': 'Мы проезжали мимо старинного',
  'word': 'сЕла',
  'right_context': ''},
 {'left_context': 'Он',
  'word': 'признаЕтся',
  'right_context': 'что давно знал об этом.'},
 {'left_context': 'Дождь',
  'word': 'стЕк',
  'right_context': 'по стеклу и исчез в водостоке.'},
 {'left_context': 'Хирург быстро',
  'word': 'отсЕк',
  'right_context': 'повреждённый участок ткани.'},
 {'left_context': 'Снег растаял, и в виде воды',
  'word': 'стЕк',
  'right_context': 'по склону.'},
 {'left_context': 'Со временем вы',
  'word': 'сознаЕте',
  'right_context': 'свою ошибку.'},
 {'left_context': 'Заводчик присвоил номер каждому',
  'word': 'помЕту',
  'right_context': ''},
 {'left_context': 'Сейчас',
  'word': 'пыхнЕт',
  'right_context': 'густым дымом.'}]

In [ ]:
def build_text(example):
    return example["left_context"] + " " + example["word"] + " " + example["right_context"]

In [ ]:
def extract_label(example):
    # в обучающем датасете должен быть правильный вариант
    # например поле "label": "ё" или "е"
    return 1 if example["label"] == "ё" else 0

In [ ]:
!pip install transformers datasets torch

In [ ]:
len(traindata)

7538

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
def count_dist(data):
  count_e = 0
  count_not_e=1
  for colocation in data:
    count_e+=colocation['target']

  return {'E': count_e, 'Ё':len(data)-count_e}

In [ ]:
count_dist(traindata)

{'E': 3832, 'Ё': 3706}

In [ ]:
import json
import random

# Config
INPUT_FILE = "yo_train.json"
SAMPLED_FILE = "sampled.json"
TRAIN_FILE = "train.json"
SAMPLE_RATIO = 0.1
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# Load data
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# Split by target
groups = {}
for item in data:
    target = item.get("target")
    groups.setdefault(target, []).append(item)

sampled_data = []
train_data = []

# Process each class separately (stratified)
for target, items in groups.items():
    n_sample = max(1, int(len(items) * SAMPLE_RATIO))

    sampled_items = random.sample(items, n_sample)

    # Use id() to safely compare objects (avoids dict equality pitfalls)
    sampled_ids = set(id(x) for x in sampled_items)

    train_items = [x for x in items if id(x) not in sampled_ids]

    sampled_data.extend(sampled_items)
    train_data.extend(train_items)

# Shuffle both sets
random.shuffle(sampled_data)
random.shuffle(train_data)

# Save outputs
with open(SAMPLED_FILE, "w", encoding="utf-8") as f:
    json.dump(sampled_data, f, ensure_ascii=False, indent=2)

with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

print(f"Sampled: {len(sampled_data)}")
print(f"Train: {len(train_data)}")

Sampled: 753
Train: 6785


In [ ]:
!pip install -U transformers

ERROR: Operation cancelled by user


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

MODEL_NAME = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

def preprocess(example):
    text = example["left_context"] + " " + example["word"] + " " + example["right_context"]
    return tokenizer(text, truncation=True, padding="max_length", max_length=128)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        text = item["left_context"] + " " + item["word"] + " " + item["right_context"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        encoding["labels"] = torch.tensor(
            item['target'],
            dtype=torch.long
        )

        return encoding

# загрузка
train_data = load_data("train.json")
val_data = load_data("sampled.json")

train_dataset = Dataset(train_data, tokenizer)
val_dataset = Dataset(val_data, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

model.to(device)  # 🔥 КЛЮЧЕВАЯ СТРОКА

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.295524,0.888446
2,0.444852,0.240648,0.933599
3,0.175384,0.267397,0.940239
4,0.104696,0.261651,0.946879
5,0.057485,0.283454,0.948207
6,0.032500,0.327422,0.945551
7,0.032500,0.310072,0.949535
8,0.018801,0.339286,0.950863
9,0.008495,0.365493,0.945551
10,0.003193,0.364490,0.946879


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4250, training_loss=0.10005338124667897, metrics={'train_runtime': 2065.7904, 'train_samples_per_second': 32.845, 'train_steps_per_second': 2.057, 'total_flos': 4463021276544000.0, 'train_loss': 0.10005338124667897, 'epoch': 10.0})

In [ ]:
def predict(model, data):
    model.eval()
    results = []

    for example in data:
        text = example["left_context"] + " " + example["word"] + " " + example["right_context"]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

        with torch.no_grad():
            inputs.to(device)
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()

        letter = "Ё" if pred == 1 else "Е"

        # заменяем Е в слове
        word = example["word"].replace("Е", letter)

        results.append({
            "left_context": example["left_context"],
            "word": word,
            "right_context": example["right_context"]
        })

    return results

In [ ]:
def predict_letter(model, data):
    model.eval()
    results = []

    for example in data:
        text = example["left_context"] + " " + example["word"] + " " + example["right_context"]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

        with torch.no_grad():
            inputs.to(device)
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()

        letter = "Ё" if pred == 1 else "Е"

        # заменяем Е в слове
        #word = example["word"].replace("Е", letter)

        results.append(letter)

    return results

In [ ]:
def save_output(data, path="output.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
test_data = load_data("yo_test.json")

predictions = predict_letter(model, test_data)



In [ ]:
def output_(strings_list):
    """
    Writes a list of strings to 'output.txt', each on a new line.

    Args:
        strings_list (list of str): The list of strings to write to the file.
    """
    with open('output.txt', 'w', encoding='utf-8') as file:
        for string in strings_list:
            file.write(string + '\n')


In [ ]:
test_data = load_data("yo_test.json")

predictions = predict_letter(model, test_data)


output_(predictions)

In [ ]:
len(predictions)

766

In [ ]:
len(testdata)

766

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

MODEL_NAME = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

def preprocess(example):
    text = example["left_context"] + " " + example["word"] + " " + example["right_context"]
    return tokenizer(text, truncation=True, padding="max_length", max_length=128)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        text = item["left_context"] + " " + item["word"] + " " + item["right_context"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        encoding["labels"] = torch.tensor(
            item['target'],
            dtype=torch.long
        )

        return encoding

# загрузка
train_data = load_data("yo_train.json")
#val_data = load_data("sampled.json")

train_dataset = Dataset(train_data, tokenizer)
#val_dataset = Dataset(val_data, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

model.to(device)  # 🔥 КЛЮЧЕВАЯ СТРОКА

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    #eval_strategy="epoch",
    #save_strategy="epoch",
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    #eval_dataset=val_dataset,
    #compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

Step,Training Loss
500,0.481119
1000,0.178579


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
test_data = load_data("yo_test.json")

predictions = predict_letter(model, test_data)


output_(predictions)
